# **Pupillometry pipeline (DeepLabCut)**
### *Track pupil area / diameter from the eye videos for the GRIN/2P rig*

On this rig the pupil/background contrast **inverts** with the visual stimulus
(bright screen → pupil looks *lighter* than the iris ring), so intensity
thresholding fails. We track pupil-edge **keypoints** with DeepLabCut and fit an
ellipse per frame.

- **Section 0** — config
- **Section 1** — extract diverse frames for labeling
- **Section 2** — DeepLabCut: create project → label → train → analyze *(GPU machine)*
- **Section 3** — keypoints → ellipse → **pupil area / diameter** (pixels + normalized) + QC
- **Section 4** — (later) align pupil to imaging frames

> Sections 0, 1, 3 need only numpy/scipy/pandas/opencv. Section 2 needs the
> `deeplabcut` package + a GPU.

## **Section 0 : Config**

In [ ]:
import os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import sys
CODE_DIR = "/Users/akhilbandi/Claude/Projects/Value Space Project"
sys.path.append(CODE_DIR)
from pupil_pipeline import PupilPipeline, PUPIL_PARTS, fit_ellipse_robust

In [ ]:
# ---- WHICH SESSION ----------------------------------------------------------
pupil_folder = "/Volumes/Runyan5/Naomi/GRINPL1/2026-06-24-pupil"
video_paths  = sorted(glob.glob(os.path.join(pupil_folder, "*.avi")))
project_dir  = os.path.join(pupil_folder, "dlc_pupil")     # DLC project lives here
camera_fps   = 10.0                                        # pupil camera frame rate

# 8 pupil-edge keypoints (clockwise). Add "reflection" (IR glint) as an extra.
pupil_parts  = list(PUPIL_PARTS)

pp = PupilPipeline(video_paths=video_paths, project_dir=project_dir,
                   pupil_parts=pupil_parts, camera_fps=camera_fps)
for v in video_paths:
    print(os.path.basename(v), pp.video_info(v))

## **Section 1 : Extract frames for labeling**

Samples frames spread across every video and re-ranks them (k-means on
appearance) so the labeling set spans **dim + bright** frames and different
pupil sizes. Streams from disk via ffmpeg — safe on the multi-GB AVIs.

In [ ]:
labeling_dir = os.path.join(project_dir, "labeling_frames")
frames = pp.extract_labeling_frames(labeling_dir, n_per_video=50, diversify=True)
print(len(frames), "frames ->", labeling_dir)

## **Section 2 : DeepLabCut (run on a GPU machine)**

**2.1 – Create the project** and set the pupil keypoints.

In [ ]:
cfg = pp.create_dlc_project(name="pupil_grinpl", experimenter="Naomi",
                            extra_parts=("reflection",))
# cfg is the path to config.yaml; keep it for the steps below
print("config:", cfg)

**2.2 – Label frames** (needs a display):

```python
import deeplabcut
deeplabcut.extract_frames(cfg, mode="automatic", userfeedback=False)  # or use ours above
deeplabcut.label_frames(cfg)
```

Labeling tips for this eye:
- Click the 8 pupil-edge points **on the pupil↔iris boundary** (N, NE, E, SE, S, SW, W, NW).
- If the eyelid covers part of the pupil, **leave the hidden points unlabeled** (DLC treats them as missing → they drop out of the ellipse fit).
- Include a few **blink / half-closed** frames so the model learns low-confidence cases.
- Optionally mark the bright **reflection** (IR glint) — useful as a stable landmark, not used in the ellipse.

**2.3 – Train & evaluate**

In [ ]:
pp.train(maxiters=200000)          # deeplabcut.train_network under the hood
import deeplabcut; deeplabcut.evaluate_network(pp.dlc_config, plotting=True)

**2.4 – Analyze the videos** → per-frame keypoints (saved as .h5 + .csv next to each video)

In [ ]:
pp.analyze(video_paths)

## **Section 3 : Keypoints → pupil area / diameter**

**3.1 – Load DLC output and fit the pupil ellipse per frame.**
Frames with too few confident pupil-edge points (blinks/occlusion) are flagged
and set to NaN; short gaps are interpolated for a clean trace.

In [ ]:
# path to a DLC output file for one video (….h5). Repeat per acquisition.
dlc_h5 = glob.glob(os.path.join(pupil_folder, "*DLC*.h5"))[0]
dlc, n = pp.load_dlc(dlc_h5)
print("frames:", n, "| bodyparts:", list(dlc.keys()))

df = pp.measure(dlc, likelihood_thr=0.6, min_points=5, smooth_win=5)
df.head()

**3.2 – QC: pupil-size trace + ellipse overlay on a sample frame**

In [ ]:
fig, ax = plt.subplots(figsize=(13,3)); pp.plot_trace(df, ax=ax); plt.show()

# overlay the fitted ellipse + keypoints on one frame (pick a good, non-blink frame)
gi = int(df.loc[df.blink==0, "frame"].iloc[len(df)//2])
vis = pp.plot_overlay(video_paths[0], dlc, gi,
                      out_png=os.path.join(project_dir, f"overlay_frame{gi}.png"))
plt.figure(figsize=(6,6)); plt.imshow(vis[...,::-1]); plt.title(f"frame {gi}"); plt.axis("off"); plt.show()

**3.3 – Save the pupil time series** (raw + normalized: `area`, `diameter`,
`area_z`, `area_fracmax`, `cx`, `cy`, `blink`).

In [ ]:
pp.save(df, os.path.join(project_dir, "pupil_trial1.csv"))

## **Section 4 : (later) Align pupil to imaging**
The pupil camera runs at ~10 Hz and imaging at ~15 Hz. To put pupil size on the
imaging frame grid we'll use the shared clock in the `.abf` (camera strobe or
common start), then resample `df.area` onto imaging-frame times — mirroring the
ball/stim alignment in `abf_imaging_alignment.py`. Deferred until the neural
side is wired up.